# LeetCode 3600 — Maximize Spanning Tree Stability with Upgrades bằng Python

- Link: https://leetcode.com/problems/maximize-spanning-tree-stability-with-upgrades/
- Độ khó: **Hard**
- Chủ đề: **Cây khung** (spanning tree), **DSU / Union-Find**, **Kruskal cây khung lớn nhất**, **binary search trên đáp án**, tham lam dùng lượt nâng cấp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxStability(self, n: int, edges: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Ôn lại **DSU** (union-find) có nén đường đi và hợp theo kích thước, dùng để phát hiện chu trình và kiểm tra liên thông.
2. Xử lý **cạnh bắt buộc**: thêm trước, nếu tạo chu trình thì không có cây khung hợp lệ (`-1`).
3. Viết hàm kiểm tra `ok(X)`: "có cây khung với mọi cạnh có sức mạnh ≥ X sau tối đa k lần nhân đôi không?" rồi **binary search** `X`.
4. Hiểu tính chất mạnh của **Kruskal cây khung lớn nhất**: dãy trọng số (đã sắp) của nó **trội từng vị trí** so với mọi cây khung khác ⇒ chỉ cần nâng cấp `k` cạnh **yếu nhất** của nó.
5. Kiểm chứng bằng vét cạn mọi tập cạnh và mọi tập nâng cấp trên đồ thị nhỏ.

## 2. Đề bài & ví dụ

Có `n` đỉnh `0..n-1` và danh sách cạnh vô hướng `edges[i] = [u, v, s, must]`:

- `s` là **sức mạnh** của cạnh.
- `must = 1`: cạnh **bắt buộc** phải có trong cây khung và **không được nâng cấp**.
- `must = 0`: cạnh tùy chọn, có thể được **nâng cấp một lần** (sức mạnh nhân đôi).

Bạn có tối đa `k` lượt nâng cấp. **Độ ổn định** của cây khung = sức mạnh **nhỏ nhất** trong các cạnh của nó (sau nâng cấp). Trả về độ ổn định **lớn nhất** có thể của một cây khung hợp lệ; nếu không có cây khung hợp lệ nào thì trả về `-1`.

**Ví dụ 1:** `n = 3, edges = [[0,1,2,1],[1,2,3,0]], k = 1` → `2` (cạnh bắt buộc sức mạnh 2 luôn nằm trong cây).

**Ví dụ 2:** `n = 3, edges = [[0,1,4,0],[1,2,3,0],[0,2,1,0]], k = 2` → `6` (chọn cạnh 4 và 3, nâng cả hai thành 8 và 6).

**Ví dụ 3:** `n = 3, edges = [[0,1,1,1],[1,2,1,1],[2,0,1,1]], k = 0` → `-1` (ba cạnh bắt buộc tạo chu trình).

**Ràng buộc:** `2 ≤ n ≤ 10^5`, `1 ≤ len(edges) ≤ 10^5`, `1 ≤ s ≤ 10^5`, `0 ≤ k ≤ n`, không có cạnh trùng, `u ≠ v`.

In [1]:
from typing import List
examples = [
    (3, [[0, 1, 2, 1], [1, 2, 3, 0]], 1, 2),
    (3, [[0, 1, 4, 0], [1, 2, 3, 0], [0, 2, 1, 0]], 2, 6),
    (3, [[0, 1, 1, 1], [1, 2, 1, 1], [2, 0, 1, 1]], 0, -1),
]
for n, edges, k, exp in examples:
    print(f"n={n}, edges={edges}, k={k} -> kỳ vọng {exp}")

n=3, edges=[[0, 1, 2, 1], [1, 2, 3, 0]], k=1 -> kỳ vọng 2
n=3, edges=[[0, 1, 4, 0], [1, 2, 3, 0], [0, 2, 1, 0]], k=2 -> kỳ vọng 6
n=3, edges=[[0, 1, 1, 1], [1, 2, 1, 1], [2, 0, 1, 1]], k=0 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Cây khung

Cây khung của đồ thị `n` đỉnh là tập `n - 1` cạnh **liên thông** mọi đỉnh và **không có chu trình**. Hai cách nói tương đương: "thêm từng cạnh, không cạnh nào nối hai đỉnh đã cùng thành phần, cuối cùng còn 1 thành phần".

### 3.2. DSU (Disjoint Set Union)

- `find(x)`: tìm đại diện thành phần của `x` (có **nén đường đi**).
- `union(a, b)`: gộp hai thành phần; trả `False` nếu `a`, `b` đã cùng thành phần (tức là thêm cạnh này sẽ tạo **chu trình**).
- Đếm số thành phần: bắt đầu `n`, mỗi lần gộp thành công giảm 1.

In [2]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.sz = [1] * n
        self.comp = n

    def find(self, x):
        p = self.p
        while p[x] != x:
            p[x] = p[p[x]]       # nén đường đi (halving)
            x = p[x]
        return x

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b:
            return False
        if self.sz[a] < self.sz[b]:
            a, b = b, a
        self.p[b] = a
        self.sz[a] += self.sz[b]
        self.comp -= 1
        return True

d = DSU(4)
print(d.union(0, 1), d.union(1, 2), d.union(0, 2), 'số thành phần =', d.comp)

True True False số thành phần = 2


### 3.3. Kruskal cho cây khung **lớn nhất**

Sắp cạnh theo trọng số **giảm dần**, lần lượt thêm cạnh nếu không tạo chu trình. Kết quả là cây khung có tổng trọng số lớn nhất — và quan trọng hơn cho bài này: nó **tối đa hóa cạnh nhỏ nhất** (bài toán "bottleneck"). Ta có thể "ép" một số cạnh vào trước (cạnh bắt buộc) rồi chạy Kruskal tiếp trên phần còn lại.

In [3]:
def kruskal_max(n, wedges):
    d = DSU(n)
    chosen = []
    for w, u, v in sorted(wedges, reverse=True):
        if d.union(u, v):
            chosen.append(w)
    return chosen if d.comp == 1 else None

print(kruskal_max(4, [(5, 0, 1), (1, 1, 2), (4, 2, 3), (3, 0, 2), (2, 1, 3)]))

[5, 4, 3]


### 3.4. Sắp xếp với `key` và tách danh sách

Ta sẽ tách cạnh thành `must` và `optional`, sắp `optional` theo `s` giảm dần: `sorted(opt, key=lambda e: -e[2])`.

In [4]:
edges = [[0, 1, 4, 0], [1, 2, 3, 0], [0, 2, 1, 0], [2, 3, 7, 1]]
must = [e for e in edges if e[3] == 1]
opt = sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2])
print('must =', must)
print('opt  =', opt)

must = [[2, 3, 7, 1]]
opt  = [[0, 1, 4, 0], [1, 2, 3, 0], [0, 2, 1, 0]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 2: ba cạnh tùy chọn có sức mạnh 4, 3, 1, `k = 2`.

- Cây khung có 2 cạnh. Các lựa chọn: `{4,3}`, `{4,1}`, `{3,1}`.
- Với `{4,3}`: nâng cả hai → `{8,6}` → độ ổn định 6.
- Với `{4,1}`: nâng cả hai → `{8,2}` → 2. Với `{3,1}`: → 2.
- Đáp án 6.

Hai nhận xét:

1. Lượt nâng cấp luôn nên dành cho **cạnh yếu nhất** trong cây (độ ổn định là min, nên tăng cạnh yếu nhất mới có ích).
2. Nên chọn cây có các cạnh **mạnh nhất có thể** — gợi ý Kruskal giảm dần.

Còn cạnh bắt buộc: chúng luôn có mặt, nên đáp án **không vượt quá** `min(s)` của cạnh bắt buộc.

In [5]:
from itertools import combinations
opts = [4, 3, 1]
for pair in combinations(opts, 2):
    up = sorted(pair)
    # nâng 2 cạnh yếu nhất (k = 2)
    print(pair, '-> sau nâng cấp', [2 * x for x in up], '-> ổn định', min(2 * x for x in up))

(4, 3) -> sau nâng cấp [6, 8] -> ổn định 6
(4, 1) -> sau nâng cấp [2, 8] -> ổn định 2
(3, 1) -> sau nâng cấp [2, 6] -> ổn định 2


## 5. Cách 1 — Brute Force: thử mọi tập cạnh và mọi tập nâng cấp

1. Duyệt mọi tập `n - 1` cạnh chứa đủ cạnh bắt buộc.
2. Kiểm tra là cây khung (DSU không gặp chu trình).
3. Duyệt mọi tập con cạnh tùy chọn (≤ k) để nâng cấp, tính min.
4. Lấy max; nếu không có cây nào → `-1`.

Chỉ dùng làm oracle với đồ thị rất nhỏ.

In [6]:
def max_stability_brute(n, edges, k):
    must_idx = {i for i, e in enumerate(edges) if e[3] == 1}
    best = -1
    for T in combinations(range(len(edges)), n - 1):
        if not must_idx <= set(T):
            continue
        d = DSU(n)
        if not all(d.union(edges[i][0], edges[i][1]) for i in T):
            continue
        opt_in_T = [i for i in T if edges[i][3] == 0]
        for r in range(0, min(k, len(opt_in_T)) + 1):
            for U in combinations(opt_in_T, r):
                U = set(U)
                val = min(edges[i][2] * (2 if i in U else 1) for i in T)
                best = max(best, val)
    return best

for n, edges, k, exp in examples:
    print(max_stability_brute(n, edges, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
6 | kỳ vọng 6
-1 | kỳ vọng -1


**Độ phức tạp:** `O(C(m, n-1) · 2^k · n)` với `m` số cạnh — chỉ dùng được với `m ≤ 10`.

**Nút thắt:** đang đoán cả cây lẫn tập nâng cấp. Hai cách gỡ: (a) cố định ngưỡng `X` và hỏi "có làm được không?" — thành bài toán liên thông đơn giản; (b) chứng minh một cây cụ thể (Kruskal lớn nhất) luôn tối ưu.

## 6. Ý tưởng thuật toán / bước đột phá

**Góc nhìn "ngưỡng" (threshold).** Cố định `X`. Một cây khung có độ ổn định `≥ X` sau ≤ k nâng cấp tồn tại khi và chỉ khi:

- Mọi cạnh bắt buộc có `s ≥ X` (chúng không nâng được), và chúng không tạo chu trình.
- Có thể nối nốt đồ thị bằng các cạnh tùy chọn, trong đó cạnh `s ≥ X` dùng **miễn phí**, cạnh `X ≤ 2s` (tức `s < X ≤ 2s`) dùng tốn **1 lượt**, cạnh `2s < X` **không dùng được**.
- Tổng số lượt ≤ k.

Để dùng ít lượt nhất: **thêm hết cạnh miễn phí trước** (DSU), sau đó mới dùng cạnh tốn phí — mỗi cạnh tốn phí chỉ dùng khi nó **nối hai thành phần khác nhau**. Số cạnh tốn phí cần dùng = số thành phần còn lại − 1, không phụ thuộc thứ tự (miễn là đồ thị liên thông được).

**Đơn điệu:** `X` đạt ⇒ mọi `X' < X` đạt. ⇒ binary search.

**Góc nhìn Kruskal (bước đột phá thứ hai).** Thêm cạnh bắt buộc, rồi Kruskal giảm dần trên cạnh tùy chọn, được các sức mạnh `w1 ≥ w2 ≥ … ≥ wm`. Đáp án là:

`min( min_must , w_{m-k} (nếu m > k) , 2·w_m (nếu m ≥ 1 và k ≥ 1) )`

— tức nâng cấp `k` cạnh yếu nhất của cây Kruskal.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi ngưỡng `X` (tuyến tính) + DSU

Đáp án luôn là một trong các giá trị `s` hoặc `2s` (độ ổn định = sức mạnh của một cạnh nào đó sau nâng cấp). Thử các ứng viên giảm dần, dừng ở ứng viên đầu tiên đạt. Đúng, nhưng có tới `2m` ứng viên, mỗi lần kiểm tra `O(m α(n))` ⇒ `O(m^2)`.

In [7]:
def check(n, edges, k, X):
    """Có cây khung với độ ổn định >= X sau <= k nâng cấp không?"""
    d = DSU(n)
    for u, v, s, m in edges:
        if m == 1:
            if s < X or not d.union(u, v):
                return False
    for u, v, s, m in edges:          # cạnh miễn phí trước
        if m == 0 and s >= X:
            d.union(u, v)
    used = 0
    for u, v, s, m in edges:          # cạnh tốn 1 lượt
        if m == 0 and s < X <= 2 * s:
            if d.union(u, v):
                used += 1
                if used > k:
                    return False
    return d.comp == 1

def has_valid_tree(n, edges):
    d = DSU(n)
    for u, v, s, m in edges:
        if m == 1 and not d.union(u, v):
            return False              # cạnh bắt buộc tạo chu trình
    for u, v, s, m in edges:
        if m == 0:
            d.union(u, v)
    return d.comp == 1

def max_stability_linear(n, edges, k):
    if not has_valid_tree(n, edges):
        return -1
    cands = sorted({s for _, _, s, _ in edges} | {2 * s for _, _, s, m in edges if m == 0}, reverse=True)
    for X in cands:
        if check(n, edges, k, X):
            return X
    return -1

for n, edges, k, exp in examples:
    print(max_stability_linear(n, edges, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
6 | kỳ vọng 6
-1 | kỳ vọng -1


Chú ý: nếu tồn tại cây khung hợp lệ thì `check` với `X = min(s)` luôn đúng (mọi cạnh miễn phí), nên vòng lặp chắc chắn trả về.

### 7.2. Cách 3 — Binary search trên `X` + DSU

Thay vòng lặp tuyến tính bằng binary search trên `[1, 2·10^5]` (hoặc trên danh sách ứng viên). Mỗi bước `O(m α(n))`, tổng `O(m α(n) log S)`.

In [8]:
def max_stability_bs(n, edges, k):
    if not has_valid_tree(n, edges):
        return -1
    lo, hi = 1, 2 * max(s for _, _, s, _ in edges)
    while lo < hi:                       # tìm X lớn nhất có check đúng
        mid = (lo + hi + 1) // 2
        if check(n, edges, k, mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for n, edges, k, exp in examples:
    print(max_stability_bs(n, edges, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
6 | kỳ vọng 6
-1 | kỳ vọng -1


Code nộp LeetCode cho cách 3:

In [9]:
class SolutionBS:
    def maxStability(self, n: int, edges: List[List[int]], k: int) -> int:
        def run(X):
            p = list(range(n))
            def find(x):
                while p[x] != x:
                    p[x] = p[p[x]]
                    x = p[x]
                return x
            comp = n
            for u, v, s, m in edges:
                if m == 1:
                    if s < X:
                        return False
                    a, b = find(u), find(v)
                    if a == b:
                        return False
                    p[a] = b; comp -= 1
            for u, v, s, m in edges:
                if m == 0 and s >= X:
                    a, b = find(u), find(v)
                    if a != b:
                        p[a] = b; comp -= 1
            used = 0
            for u, v, s, m in edges:
                if m == 0 and s < X <= 2 * s:
                    a, b = find(u), find(v)
                    if a != b:
                        used += 1
                        if used > k:
                            return False
                        p[a] = b; comp -= 1
            return comp == 1

        if not run(1):
            return -1
        lo, hi = 1, 2 * max(e[2] for e in edges)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if run(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print([SolutionBS().maxStability(n, e, k) for n, e, k, _ in examples])

[2, 6, -1]


Lưu ý `run(1)`: với `X = 1` mọi cạnh đều miễn phí, nên `run(1)` đúng ⇔ có cây khung hợp lệ.

### 7.3. Cách 4 — Kruskal cây khung lớn nhất + nâng `k` cạnh yếu nhất (lời giải chính)

1. DSU, thêm mọi cạnh bắt buộc; gặp chu trình → `-1`. Ghi `must_min`.
2. Sắp cạnh tùy chọn theo `s` giảm dần, Kruskal: cạnh nào nối hai thành phần thì nhận, lưu sức mạnh vào `chosen` (tự nhiên đã giảm dần).
3. Nếu chưa liên thông → `-1`.
4. Nâng cấp `min(k, len(chosen))` cạnh cuối của `chosen`. Độ ổn định = min của tất cả.

In [10]:
def max_stability_kruskal(n, edges, k):
    d = DSU(n)
    INF = float('inf')
    must_min = INF
    for u, v, s, m in edges:
        if m == 1:
            if not d.union(u, v):
                return -1
            must_min = min(must_min, s)
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            chosen.append(s)
    if d.comp != 1:
        return -1
    ans = must_min
    t = min(k, len(chosen))
    for i, s in enumerate(chosen):
        val = 2 * s if i >= len(chosen) - t else s
        ans = min(ans, val)
    return ans

for n, edges, k, exp in examples:
    print(max_stability_kruskal(n, edges, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
6 | kỳ vọng 6
-1 | kỳ vọng -1


**Độ phức tạp:** sắp xếp `O(m log m)` + DSU `O(m α(n))`. Space `O(n + m)`.

### 7.4. Cách 5 — Kruskal với công thức đóng

Vì `chosen` giảm dần, chỉ có hai vị trí ảnh hưởng: phần tử cuối (`w_m`, sau nâng thành `2·w_m`) và phần tử cuối cùng **không** được nâng (`w_{m-k}`). Không cần vòng lặp:

In [11]:
def finish(must_min, chosen, k):
    m = len(chosen)
    ans = must_min
    if m:
        if k >= 1:
            ans = min(ans, 2 * chosen[-1])
        else:
            ans = min(ans, chosen[-1])
        if m > k:
            ans = min(ans, chosen[m - k - 1])
    return ans

print(finish(float('inf'), [4, 3], 2))   # 6
print(finish(float('inf'), [4, 3], 1))   # min(2*3, 4) = 4
print(finish(2, [3], 1))                 # 2

6
4
2


## 8. TRACE từng bước

Đồ thị 5 đỉnh, `k = 2`:

- Bắt buộc: `(0,1,9)`.
- Tùy chọn: `(1,2,5)`, `(2,3,3)`, `(3,4,4)`, `(1,3,6)`, `(0,4,2)`, `(2,4,7)`.

Bảng dự kiến (Kruskal giảm dần trên cạnh tùy chọn sau khi đã nối 0–1):

| bước | cạnh | s | nối 2 thành phần? | chosen |
|---|---|---|---|---|
| 1 | (2,4) | 7 | có | [7] |
| 2 | (1,3) | 6 | có | [7,6] |
| 3 | (1,2) | 5 | có | [7,6,5] |
| 4 | (3,4) | 4 | không (đã nối qua 3-1-2-4) | |
| ... | | | | |

Nâng 2 cạnh yếu nhất `6, 5` → `12, 10`; min với `7` và cạnh bắt buộc `9` → **7**.

In [12]:
def trace_kruskal(n, edges, k):
    d = DSU(n)
    must_min = float('inf')
    for u, v, s, m in edges:
        if m == 1:
            ok = d.union(u, v)
            print(f"[bắt buộc] ({u},{v}) s={s} -> {'nhận' if ok else 'CHU TRÌNH -> -1'}")
            if not ok:
                return -1
            must_min = min(must_min, s)
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        ok = d.union(u, v)
        if ok:
            chosen.append(s)
        print(f"[tùy chọn] ({u},{v}) s={s} -> {'nhận' if ok else 'bỏ (chu trình)'}; "
              f"chosen={chosen}; số thành phần={d.comp}")
    if d.comp != 1:
        print('Không liên thông -> -1')
        return -1
    t = min(k, len(chosen))
    final = chosen[:len(chosen) - t] + [2 * s for s in chosen[len(chosen) - t:]]
    print('Nâng cấp', t, 'cạnh yếu nhất ->', final, '| must_min =', must_min)
    ans = min([must_min] + final)
    print('Đáp án =', ans)
    return ans

g = [[0, 1, 9, 1], [1, 2, 5, 0], [2, 3, 3, 0], [3, 4, 4, 0], [1, 3, 6, 0], [0, 4, 2, 0], [2, 4, 7, 0]]
trace_kruskal(5, g, 2)

[bắt buộc] (0,1) s=9 -> nhận
[tùy chọn] (2,4) s=7 -> nhận; chosen=[7]; số thành phần=3
[tùy chọn] (1,3) s=6 -> nhận; chosen=[7, 6]; số thành phần=2
[tùy chọn] (1,2) s=5 -> nhận; chosen=[7, 6, 5]; số thành phần=1
[tùy chọn] (3,4) s=4 -> bỏ (chu trình); chosen=[7, 6, 5]; số thành phần=1
[tùy chọn] (2,3) s=3 -> bỏ (chu trình); chosen=[7, 6, 5]; số thành phần=1
[tùy chọn] (0,4) s=2 -> bỏ (chu trình); chosen=[7, 6, 5]; số thành phần=1
Nâng cấp 2 cạnh yếu nhất -> [7, 12, 10] | must_min = 9
Đáp án = 7


7

Đối chiếu với cách binary search trên cùng đồ thị: in ra `check(X)` quanh đáp án để thấy tính đơn điệu.

In [13]:
def trace_check(n, edges, k, Xs):
    for X in Xs:
        print(f"X={X:>2}: check = {check(n, edges, k, X)}")

trace_check(5, g, 2, range(1, 13))
print('vét cạn:', max_stability_brute(5, g, 2))

X= 1: check = True
X= 2: check = True
X= 3: check = True
X= 4: check = True
X= 5: check = True
X= 6: check = True
X= 7: check = True
X= 8: check = False
X= 9: check = False
X=10: check = False
X=11: check = False
X=12: check = False
vét cạn: 7


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxStability(self, n: int, edges: List[List[int]], k: int) -> int:
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        comp = n
        must_min = float('inf')
        for u, v, s, m in edges:
            if m == 1:
                a, b = find(u), find(v)
                if a == b:
                    return -1             # cạnh bắt buộc tạo chu trình
                parent[a] = b
                comp -= 1
                must_min = min(must_min, s)

        chosen = []
        for u, v, s, m in sorted(edges, key=lambda e: -e[2]):
            if m == 0:
                a, b = find(u), find(v)
                if a != b:
                    parent[a] = b
                    comp -= 1
                    chosen.append(s)
        if comp != 1:
            return -1

        ans = must_min
        c = len(chosen)
        if c:
            ans = min(ans, 2 * chosen[-1] if k >= 1 else chosen[-1])
            if c > k:
                ans = min(ans, chosen[c - k - 1])
        return ans

for n, edges, k, exp in examples:
    got = Solution().maxStability(n, edges, k)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
6 OK
-1 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (tính trội của Kruskal).** Gọi `T*` là cây Kruskal giảm dần (sau khi ép cạnh bắt buộc) với các cạnh tùy chọn có sức mạnh `w1 ≥ … ≥ wm`. Với mọi cây khung hợp lệ khác `T` (chứa đủ cạnh bắt buộc) có cạnh tùy chọn `t1 ≥ … ≥ tm` (cùng số lượng `m = n - 1 - #bắt buộc`), ta có `wi ≥ ti` với mọi `i`.

*Lý do:* với mọi ngưỡng `Y`, số cạnh `≥ Y` mà Kruskal chọn bằng số cạnh rừng lớn nhất dùng được trong đồ thị con các cạnh `≥ Y` (đã có cạnh bắt buộc) — đây là tính chất matroid: tham lam đạt hạng tối đa trên mỗi "tầng" ngưỡng. Cây `T` không thể có nhiều cạnh `≥ Y` hơn hạng đó. Vậy với mọi `Y`, `#{i: wi ≥ Y} ≥ #{i: ti ≥ Y}` ⇔ `wi ≥ ti` từng vị trí.

**Nâng cấp tối ưu trên một cây cố định.** Độ ổn định là `min`. Nếu có lượt nâng cấp chưa dùng cho cạnh yếu nhất nhưng dùng cho cạnh khác, chuyển lượt đó sang cạnh yếu nhất không làm min giảm. Vậy nâng `k` cạnh yếu nhất là tối ưu, cho giá trị `f(t) = min(must_min, t_{m-k}, 2·t_m)` — hàm **không giảm** theo từng `ti`.

Kết hợp: `f(w) ≥ f(t)` với mọi cây `T` ⇒ cây Kruskal cho đáp án tối ưu.

**Trường hợp `-1`:** cạnh bắt buộc tạo chu trình (không cây nào chứa đủ chúng), hoặc toàn đồ thị không liên thông.

**Cách 3 đúng** vì `check(X)` đặc trưng chính xác "tồn tại cây ≥ X": cạnh `2s < X` vô dụng, cạnh `s ≥ X` miễn phí nên thêm hết để giảm số thành phần, mỗi cạnh tốn phí giảm đúng 1 thành phần; số lượt tối thiểu = (số thành phần sau cạnh miễn phí) − 1.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(edges)`, `S = max(s) ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(C(m, n-1) · 2^k · n)` | `O(n)` |
| 2. Thử mọi ngưỡng | `O(m^2 α(n))` | `O(n + m)` |
| 3. Binary search + DSU | `O(m α(n) log S)` | `O(n)` |
| 4/5. Kruskal | `O(m log m + m α(n))` | `O(n + m)` |

`α(n)` là hàm Ackermann ngược, coi như hằng số.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [15]:
impls = {
    'linear': max_stability_linear,
    'bs': max_stability_bs,
    'bs_submit': lambda n, e, k: SolutionBS().maxStability(n, e, k),
    'kruskal': max_stability_kruskal,
    'submit': lambda n, e, k: Solution().maxStability(n, e, k),
}
edge_cases = [
    (2, [[0, 1, 5, 1]], 0, 5),           # chỉ một cạnh bắt buộc
    (2, [[0, 1, 5, 0]], 1, 10),          # nâng cạnh duy nhất
    (2, [[0, 1, 5, 0]], 0, 5),
    (3, [[0, 1, 5, 0]], 3, -1),          # không liên thông
    (3, [[0, 1, 1, 1], [1, 2, 9, 0]], 1, 1),   # cạnh bắt buộc yếu chặn trên đáp án
    (4, [[0, 1, 2, 1], [2, 3, 3, 1], [1, 2, 100, 0]], 0, 2),
    (3, [[0, 1, 3, 0], [1, 2, 4, 0], [0, 2, 7, 0]], 1, 7),  # chọn {7,4}, nâng 4 -> 8
]
for n, e, k, exp in examples + edge_cases:
    res = {name: f(n, e, k) for name, f in impls.items()}
    assert all(v == exp for v in res.values()), (n, e, k, res, exp)
    assert max_stability_brute(n, e, k) == exp, (n, e, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [16]:
import random
random.seed(3600)

def rand_graph(n, m, smax=8, pmust=0.25):
    pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]
    random.shuffle(pairs)
    es = []
    for u, v in pairs[:m]:
        if random.random() < 0.5:
            u, v = v, u
        es.append([u, v, random.randint(1, smax), 1 if random.random() < pmust else 0])
    return es

cnt = 0
for _ in range(700):
    n = random.randint(2, 5)
    m = random.randint(1, min(8, n * (n - 1) // 2))
    e = rand_graph(n, m)
    k = random.randint(0, n)
    exp = max_stability_brute(n, e, k)
    for name, f in impls.items():
        got = f(n, e, k)
        assert got == exp, (name, n, e, k, got, exp)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'đồ thị ngẫu nhiên.')

Khớp vét cạn trên 700 đồ thị ngẫu nhiên.


### 12.3. Ca lớn (`5·10^4` đỉnh, `m = 10^5` cạnh)

In [17]:
import time
def big_graph(n, m, pmust):
    es = []
    # cây ngẫu nhiên để đảm bảo liên thông, cạnh bắt buộc lấy từ cây (không tạo chu trình)
    for v in range(1, n):
        u = random.randrange(v)
        es.append([u, v, random.randint(1, 10**5), 1 if random.random() < pmust else 0])
    seen = {(min(a, b), max(a, b)) for a, b, _, _ in es}
    while len(es) < m:
        u, v = random.randrange(n), random.randrange(n)
        if u != v and (min(u, v), max(u, v)) not in seen:
            seen.add((min(u, v), max(u, v)))
            es.append([u, v, random.randint(1, 10**5), 0])
    return es

random.seed(11)
n = 10**5
for pmust, k in [(0.0, n), (0.3, 1000), (0.0, 0)]:
    e = big_graph(n // 2, n, pmust)       # 5·10^4 đỉnh, 10^5 cạnh
    t = time.time(); a = Solution().maxStability(n // 2, e, k); t1 = time.time() - t
    t = time.time(); b = SolutionBS().maxStability(n // 2, e, k); t2 = time.time() - t
    assert a == b
    print(f"pmust={pmust}, k={k}: đáp án {a} | Kruskal {t1:.2f}s | binary search {t2:.2f}s")

pmust=0.0, k=100000: đáp án 68 | Kruskal 0.13s | binary search 1.20s


pmust=0.3, k=1000: đáp án 1 | Kruskal 0.15s | binary search 0.11s


pmust=0.0, k=0: đáp án 11 | Kruskal 0.14s | binary search 1.23s


Kruskal chạy cỡ 0,1–0,2 s; binary search chậm hơn nhiều lần (khoảng 18 lần DSU toàn bộ cạnh) nhưng vẫn trong vài giây.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép nâng cấp cả cạnh bắt buộc

In [18]:
def wrong_upgrade_must(n, edges, k):
    # SAI: coi mọi cạnh như nhau, chạy Kruskal rồi nâng k cạnh yếu nhất
    d = DSU(n)
    chosen = []
    for u, v, s, m in edges:
        if m == 1 and d.union(u, v):
            chosen.append(s)
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            chosen.append(s)
    chosen.sort(reverse=True)
    t = min(k, len(chosen))
    final = chosen[:len(chosen) - t] + [2 * x for x in chosen[len(chosen) - t:]]
    return min(final) if d.comp == 1 else -1

ex = (3, [[0, 1, 2, 1], [1, 2, 3, 0]], 1)
print('sai :', wrong_upgrade_must(*ex))      # 3 (nâng cạnh bắt buộc 2 -> 4)
print('đúng:', Solution().maxStability(*ex)) # 2

sai : 3
đúng: 2


**Vì sao sai:** đề nói cạnh bắt buộc **không được nâng cấp**; `must_min` là chặn trên cứng của đáp án.

### Lỗi 2 — Quên kiểm tra chu trình giữa các cạnh bắt buộc

In [19]:
def wrong_no_cycle_check(n, edges, k):
    d = DSU(n)
    must_min = float('inf')
    for u, v, s, m in edges:
        if m == 1:
            d.union(u, v)                  # SAI: bỏ qua kết quả False
            must_min = min(must_min, s)
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            chosen.append(s)
    if d.comp != 1:
        return -1
    return finish(must_min, chosen, k)

ex = (3, [[0, 1, 1, 1], [1, 2, 1, 1], [2, 0, 1, 1]], 0)
print('sai :', wrong_no_cycle_check(*ex))    # 1
print('đúng:', Solution().maxStability(*ex)) # -1

sai : 1
đúng: -1


**Vì sao sai:** ba cạnh bắt buộc tạo chu trình, không cây khung nào chứa cả ba ⇒ phải trả `-1`.

### Lỗi 3 — Nâng cấp các cạnh **mạnh nhất** (đầu danh sách) thay vì yếu nhất

In [20]:
def wrong_upgrade_strongest(n, edges, k):
    d = DSU(n)
    must_min = float('inf')
    for u, v, s, m in edges:
        if m == 1:
            if not d.union(u, v):
                return -1
            must_min = min(must_min, s)
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            chosen.append(s)
    if d.comp != 1:
        return -1
    final = [2 * x for x in chosen[:k]] + chosen[k:]   # SAI
    return min([must_min] + final)

ex = (3, [[0, 1, 4, 0], [1, 2, 3, 0], [0, 2, 1, 0]], 1)
print('sai :', wrong_upgrade_strongest(*ex))  # 3
print('đúng:', Solution().maxStability(*ex))  # 4

sai : 3
đúng: 4


**Vì sao sai:** min chỉ tăng khi tăng phần tử nhỏ nhất. `chosen` sắp **giảm dần**, nên phần tử yếu nằm ở **cuối**.

### Lỗi 4 — Dùng Kruskal **tăng dần** (cây khung nhỏ nhất)

In [21]:
def wrong_min_spanning(n, edges, k):
    d = DSU(n)
    must_min = float('inf')
    for u, v, s, m in edges:
        if m == 1:
            if not d.union(u, v):
                return -1
            must_min = min(must_min, s)
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: e[2]):  # SAI
        if d.union(u, v):
            chosen.append(s)
    if d.comp != 1:
        return -1
    return finish(must_min, sorted(chosen, reverse=True), k)

ex = examples[1][:3]
print('sai :', wrong_min_spanning(*ex))       # 2
print('đúng:', Solution().maxStability(*ex))  # 6

sai : 2
đúng: 6


**Vì sao sai:** ta muốn cạnh **mạnh** nhất, nên Kruskal phải sắp **giảm dần**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Max của min** trên cây khung" ⇒ cây khung lớn nhất (Kruskal giảm dần) / bài toán bottleneck.
- "Cạnh bắt buộc" ⇒ union trước, kiểm tra chu trình ngay.
- "Có thể nâng cấp tối đa k phần tử, tối đa hóa min" ⇒ luôn dồn nâng cấp cho **phần tử yếu nhất**.
- Không chắc về tính chất trội? Dùng **binary search + kiểm tra liên thông** — an toàn, dễ chứng minh, chỉ chậm hơn một hệ số `log`.
- Kiểm tra `-1` có hai nguồn: chu trình cạnh bắt buộc và đồ thị không liên thông.

## 15. Biến thể của bài

1. Mỗi lần nâng cấp **cộng thêm `d`** thay vì nhân đôi — cùng khung, `check` dùng điều kiện `s + d ≥ X`.
2. Cho phép nâng cấp nhiều lần (nhân `2^t`) với tổng `k` lượt ⇒ `check` tính số lượt cần cho mỗi cạnh = `ceil(log2(X/s))`, nhưng khi đó cạnh tốn phí có chi phí khác nhau ⇒ dùng Kruskal theo chi phí tăng dần.
3. Tối đa hóa **tổng** thay vì min ⇒ cây khung lớn nhất rồi nâng `k` cạnh **mạnh nhất** (nhân đôi lợi nhiều nhất).
4. LC 1489 *Find Critical and Pseudo-Critical Edges in MST*, LC 1135 *Connecting Cities With Minimum Cost*, LC 1631 *Path With Minimum Effort* (bottleneck + binary search/DSU).

In [22]:
# Minh hoạ biến thể 3: tối đa hoá TỔNG thay vì min
def max_total(n, edges, k):
    d = DSU(n)
    total = 0
    for u, v, s, m in edges:
        if m == 1:
            if not d.union(u, v):
                return -1
            total += s
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            chosen.append(s)
    if d.comp != 1:
        return -1
    return total + sum(chosen) + sum(chosen[:k])   # nhân đôi k cạnh mạnh nhất

print(max_total(5, g, 2))

40


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn cây + nâng cấp | mũ | `O(n)` | Oracle kiểm thử |
| 2 | Thử mọi ngưỡng giảm dần | `O(m^2)` | `O(n+m)` | Hiểu hàm `check` |
| 3 | Binary search + DSU | `O(m log S)` | `O(n)` | An toàn, dễ chứng minh |
| 4 | Kruskal + nâng k cạnh yếu | `O(m log m)` | `O(n+m)` | **Nộp bài** |
| 5 | Kruskal + công thức đóng | `O(m log m)` | `O(n+m)` | Gọn nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n=4, edges=[[0,1,5,0],[1,2,5,0],[2,3,1,0]], k=1` → ?

**Bài 2 (tính tay):** Với đồ thị `g` ở mục 8 nhưng `k = 0`, đáp án là bao nhiêu?

**Bài 3 (điền TODO):** hoàn thành `check_todo(n, edges, k, X)`.

**Bài 4 (tự viết lại):** viết `max_stability_add(n, edges, k, d)` cho biến thể "nâng cấp = cộng thêm d".

**Bài 5 (mini challenge):** trả về **danh sách cạnh** của cây tối ưu và tập cạnh được nâng cấp.

In [23]:
def check_todo(n, edges, k, X):
    # TODO: thêm cạnh bắt buộc (kiểm tra s >= X và không chu trình)
    # TODO: thêm cạnh miễn phí, rồi cạnh tốn 1 lượt, đếm lượt
    raise NotImplementedError

def max_stability_add_todo(n, edges, k, d):
    # TODO
    raise NotImplementedError

def best_tree_todo(n, edges, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cây phải dùng cả 3 cạnh. (2) Nâng cạnh yếu nhất 1 → 2. (3) min(5,5,2) = 2.

**Bài 2.** (1) Kruskal chọn 7, 6, 5 (như mục 8). (2) Không nâng cấp. (3) min(9, 7, 6, 5) = 5.

**Bài 3.** (1) Bốn vòng lặp như mục 7.1. (2) Đừng quên `s < X` với cạnh bắt buộc. (3) Cuối cùng `comp == 1`.

**Bài 4.** (1) Kruskal giữ nguyên. (2) Nâng k cạnh yếu nhất: `s + d`. (3) Công thức `min(must_min, w_{m-k}, w_m + d)`.

**Bài 5.** (1) Lưu `(u, v, s)` thay vì chỉ `s`. (2) k phần tử cuối là cạnh nâng cấp. (3) Kiểm tra lại bằng DSU rằng đó là cây khung.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, Bài 2
a1 = Solution().maxStability(4, [[0, 1, 5, 0], [1, 2, 5, 0], [2, 3, 1, 0]], 1)
a2 = Solution().maxStability(5, g, 0)
assert a1 == 2 and a2 == 5
assert max_stability_brute(5, g, 0) == 5
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 2 | Bài 2: 5


In [25]:
# Bài 3
def check_ans(n, edges, k, X):
    d = DSU(n)
    for u, v, s, m in edges:
        if m == 1 and (s < X or not d.union(u, v)):
            return False
    for u, v, s, m in edges:
        if m == 0 and s >= X:
            d.union(u, v)
    used = 0
    for u, v, s, m in edges:
        if m == 0 and s < X <= 2 * s and d.union(u, v):
            used += 1
    return used <= k and d.comp == 1

random.seed(5)
for _ in range(300):
    n = random.randint(2, 5)
    e = rand_graph(n, random.randint(1, min(8, n * (n - 1) // 2)))
    k = random.randint(0, n)
    for X in range(1, 18):
        assert check_ans(n, e, k, X) == check(n, e, k, X)
print('Bài 3: check_ans khớp check')

Bài 3: check_ans khớp check


In [26]:
# Bài 4: nâng cấp = cộng d
def max_stability_add(n, edges, k, dd):
    d = DSU(n)
    must_min = float('inf')
    for u, v, s, m in edges:
        if m == 1:
            if not d.union(u, v):
                return -1
            must_min = min(must_min, s)
    chosen = []
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            chosen.append(s)
    if d.comp != 1:
        return -1
    t = min(k, len(chosen))
    final = chosen[:len(chosen) - t] + [x + dd for x in chosen[len(chosen) - t:]]
    return min([must_min] + final)

def brute_add(n, edges, k, dd):
    must_idx = {i for i, e in enumerate(edges) if e[3] == 1}
    best = -1
    for T in combinations(range(len(edges)), n - 1):
        if not must_idx <= set(T):
            continue
        d = DSU(n)
        if not all(d.union(edges[i][0], edges[i][1]) for i in T):
            continue
        opt = sorted(edges[i][2] for i in T if edges[i][3] == 0)
        mus = [edges[i][2] for i in T if edges[i][3] == 1]
        t = min(k, len(opt))
        vals = mus + [x + dd for x in opt[:t]] + opt[t:]
        best = max(best, min(vals))
    return best

random.seed(6)
for _ in range(300):
    n = random.randint(2, 5)
    e = rand_graph(n, random.randint(1, min(8, n * (n - 1) // 2)))
    k, dd = random.randint(0, n), random.randint(0, 5)
    assert max_stability_add(n, e, k, dd) == brute_add(n, e, k, dd)
print('Bài 4: khớp vét cạn')

Bài 4: khớp vét cạn


In [27]:
# Bài 5: trả về cây tối ưu
def best_tree(n, edges, k):
    d = DSU(n)
    tree, opt = [], []
    for u, v, s, m in edges:
        if m == 1:
            if not d.union(u, v):
                return None
            tree.append((u, v, s))
    for u, v, s, m in sorted((e for e in edges if e[3] == 0), key=lambda e: -e[2]):
        if d.union(u, v):
            opt.append((u, v, s))
    if d.comp != 1:
        return None
    t = min(k, len(opt))
    upgraded = opt[len(opt) - t:]
    return tree + opt, upgraded

tree, up = best_tree(5, g, 2)
d = DSU(5)
assert len(tree) == 4 and all(d.union(u, v) for u, v, _ in tree)
val = min([s for u, v, s in tree if (u, v, s) not in up] + [2 * s for _, _, s in up])
assert val == Solution().maxStability(5, g, 2)
print('Cây:', tree, '| nâng cấp:', up, '| độ ổn định:', val)

Cây: [(0, 1, 9), (2, 4, 7), (1, 3, 6), (1, 2, 5)] | nâng cấp: [(1, 3, 6), (1, 2, 5)] | độ ổn định: 7


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
DSU; must_min = inf
for cạnh bắt buộc: union, gặp chu trình -> return -1; must_min = min(must_min, s)
chosen = [s của cạnh tùy chọn được Kruskal (s giảm dần) nhận]
nếu chưa liên thông -> -1
nâng min(k, len(chosen)) cạnh CUỐI (yếu nhất) của chosen
return min(must_min, các cạnh sau nâng cấp)
```

**Checklist:**

- [ ] Cạnh bắt buộc: thêm trước, kiểm tra chu trình, không nâng cấp?
- [ ] Kruskal sắp **giảm dần**?
- [ ] Nâng cấp cạnh **yếu nhất** (cuối danh sách)?
- [ ] Kiểm tra liên thông cuối cùng (`comp == 1`)?
- [ ] Trường hợp không có cạnh tùy chọn trong cây (chỉ cạnh bắt buộc)?
- [ ] Đã so với vét cạn và binary search?